<a href="https://colab.research.google.com/github/TristanNguyen2311/Retail-Customer-Segmentation-with-RFM-Analysis-Python/blob/main/RFM_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CONTEXT**
RFM là một mô hình phân tích khách hàng dựa trên ba yếu tố chính:

Recency – R (Thời gian mua hàng gần nhất): Khoảng thời gian kể từ lần giao dịch gần nhất của khách hàng.

Frequency – F (Tần suất mua hàng): Số lần giao dịch khách hàng đã thực hiện trong khoảng thời gian nhất định.

Monetary – M (Giá trị tiền cho mỗi lần mua hàng): Tổng giá trị của các giao dịch mà khách hàng đã thực hiện trong một khoảng thời gian nhất định.

RFM là một phần của Marketing Analysis và được sử dụng để phân
tích giá trị khách hàng (Customer Value), giúp doanh nghiệp có thể phân tích ra từng nhóm khách
hàng mà mình đang có. Từ đó có những chiến dịch marketing hoặc chăm sóc đặc biệt.

# **PROBLEM STATEMENT**
Công ty SuperStore là một công ty bán lẻ trên toàn cầu - Global. Nên công ty có rất nhiều khách hàng.
Nhân dịp giáng sinh và năm mới, phòng Marketing muốn chạy các chiến dịch marketing để tri ân khách hàng đã ủng hộ công ty suốt thời gian qua. Cũng như khai thác các khách có tiềm năng trở thành khách hàng trung thành.

Tuy nhiên phòng Marketing vẫn chưa phân nhóm cho từng khách hàng của năm nay được vì tập dữ liệu quá lớn nên không thể xử lý bằng tay như các năm trước, nên nhờ Phòng Phân tích dữ liệu hỗ trợ triển khai một bài toán phân loại phân khúc của từng khách hàng để triển khai từng chương trình marketing phù hợp với từng nhóm khách hàng.

Giám đốc Marketing cũng có đề xuất phương án sử dụng mô hình RFM, tuy nhiên trước đây khi quy mô công ty nhỏ, team có thể tự tính và phân loại bằng excel. Hiện tại lượng data quá lớn nên mong muốn Phòng dữ liệu xây dựng luồng triển khai đánh giá Segmentation thông qua lập trình Python.

# **USER PROFILE DEFINITION**

## **Part 1: Exploratory Data Analysis (EDA)**

*   Understand about the data (data type, data value)
*   Check missing data type and handle missing data
*   Check duplicated data type and handle duplicated data
*   Check outliers and handle outliers












### 1.1 Understand about the data (data type, data value)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/Unigap/Python/RFM Segmentation'

### Tự động cập nhật đường dẫn tệp

Để giúp notebook linh hoạt hơn khi bạn di chuyển tệp `ecommerce retail.csv` trong Google Drive, chúng ta có thể thêm một hàm tìm kiếm. Hàm này sẽ cố gắng tìm tệp trong `MyDrive` nếu đường dẫn mặc định không hoạt động. Khi tìm thấy, nó sẽ cập nhật biến `path` để các phần còn lại của notebook có thể sử dụng đường dẫn chính xác.

In [ ]:
import os

def find_file_in_drive(filename, search_start_path='/content/drive/MyDrive'):
    """
    Searches for a specified file within the user's Google Drive.

    Args:
        filename (str): The name of the file to search for.
        search_start_path (str): The starting directory for the search (default is MyDrive).

    Returns:
        str or None: The absolute path to the file if found, otherwise None.
    """
    for root, dirs, files in os.walk(search_start_path):
        if filename in files:
            return os.path.join(root, filename)
    return None


In [ ]:
# Load Dataset
import pandas as pd
ecommerce_retail = pd.read_csv("/content/drive/MyDrive/Unigap/Python/RFM Segmentation/ecommerce retail.csv", encoding='latin1')
ecommerce_retail.head()

In [ ]:
# Detect the data type of each column
ecommerce_retail.info()

In [ ]:
# Convert data type
ecommerce_retail['InvoiceNo']= ecommerce_retail['InvoiceNo'].astype('string')
ecommerce_retail['StockCode']= ecommerce_retail['StockCode'].astype('string')
ecommerce_retail['Description']= ecommerce_retail['Description'].astype('string')
ecommerce_retail['InvoiceDate']= pd.to_datetime(ecommerce_retail['InvoiceDate'])
ecommerce_retail['CustomerID']= ecommerce_retail['CustomerID'].astype('string')
ecommerce_retail['Country']= ecommerce_retail['Country'].astype('string')

In [ ]:
ecommerce_retail.shape

In [ ]:
# Detect data value of columns
ecommerce_retail.describe()

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport # Import the ProfileReport function


In [ ]:
profile = ProfileReport(ecommerce_retail)
profile

In [ ]:
# Check data category data types of column StockCode
stockcode_check = ecommerce_retail['StockCode'].value_counts()
stockcode_check

In [ ]:
# Check data category data types of column Description
description_check = ecommerce_retail['Description'].value_counts()
description_check

In [ ]:
description_check.to_csv(path + '/description_check.csv')

In [ ]:
description_check_update = pd.read_csv(path + '/description_check.csv')

# Add column 'Error': True if any letter is not uppercase or contains only '?
description_check_update['Error'] = description_check_update['Description'].str.contains(r'[a-z]|\?', regex=True)
description_check_update

In [ ]:
ecommerce_retail_update = ecommerce_retail.merge(description_check_update[['Description', 'Error']], on='Description', how='left')
print(ecommerce_retail_update[ecommerce_retail_update['Error'] == True].shape)
print(ecommerce_retail_update.shape)

In [ ]:
# Check the reason for data quantity <0
ecommerce_retail_update[ecommerce_retail_update['Quantity'] < 0].head()


In [ ]:
# Check if Quantity <0 is due to cancellation
ecommerce_retail_update[ecommerce_retail_update['InvoiceNo'].str.startswith('C') & (ecommerce_retail_update['Quantity'] < 0)].head()

In [ ]:
# Check if Quantity <0 that is not due to cancellation
ecommerce_retail_update[~ecommerce_retail_update['InvoiceNo'].str.startswith('C') & (ecommerce_retail_update['Quantity'] < 0)].head()

In [ ]:
# Check the reason for Unit Price <0
ecommerce_retail_update[ecommerce_retail_update['UnitPrice'] < 0]

In [ ]:
# Check data types
ecommerce_retail_update.dtypes

**Nhận xét:**
- Có các cột chưa đúng data type nên convert về đúng dạng
- Có missing values ở cột Description và cột CustomerID
- Cột Description có 3092 đơn hàng có nội dung mô tả không chính xác
- Gần 88% các trường hợp có Quantity <0 là do bị cancle, 12% các trường hợp còn lại đến từ các lí do như: thất lạc, hư hỏng, đang kiểm tra lại hoặc chưa có thông tin và ta có thể thấy UnitPrice = 0
- 2 trường hợp có UnitPrice < 0 là do điều chỉnh nợ xấu

### 1.2 Handle incorrect values

In [ ]:
# Remove orders with Quantity <=0 (including canceled orders)
ecommerce_retail_update = ecommerce_retail_update[ecommerce_retail_update['Quantity'] > 0]
ecommerce_retail_update.shape


In [ ]:
# Remove orders with UnitPrice <=0
ecommerce_retail_update = ecommerce_retail_update[ecommerce_retail_update['UnitPrice'] > 0]
ecommerce_retail_update.shape

### 1.3 Handle missing values

In [ ]:
# Check missing value
missing_values = ecommerce_retail_update.isnull().sum()
missing_percentage = (ecommerce_retail_update.isnull().sum() / len(ecommerce_retail_update)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_values, 'Missing Percentage': missing_percentage})
missing_df

In [ ]:
# Check the rows with missing CustomerID to understand the reason
ecommerce_retail_update[ecommerce_retail_update['CustomerID'].isna()].head()

In [ ]:
# Create a month column to check for missing values by month
ecommerce_retail_update['day'] = ecommerce_retail_update['InvoiceDate'].dt.date
ecommerce_retail_update['month'] = ecommerce_retail_update['InvoiceDate'].dt.strftime('%Y-%m')
ecommerce_retail_update.head()

In [ ]:
ecommerce_retail_update[ecommerce_retail_update['CustomerID'].isna()]['month'].value_counts().sort_index()

In [ ]:
# Remove missing values in the CustomerID column
ecommerce_retail_update = ecommerce_retail_update[ecommerce_retail_update['CustomerID'].notnull()]
ecommerce_retail_update.shape

**Nhận xét**

- Cột CustomerID có 132220 missing value (chiếm gần 25%)
- Tháng nào cũng đều có missing value do đó cần check lại hệ thống hay quy trình lưu trữ data để khắc phục
- Customer ID là dữ liệu quan trọng không thể thay thế nên chỉ có thể xóa đi

### 1.4 Handle duplicate values

In [ ]:
# Check duplicate values
duplicates_df= ecommerce_retail_update.duplicated(subset=['InvoiceNo','StockCode','InvoiceDate','CustomerID'])
ecommerce_retail_update[duplicates_df].head()

In [ ]:
# Check specific InvoiceNos that are duplicated
ecommerce_retail_update[(ecommerce_retail_update['InvoiceNo'] == '536381') & (ecommerce_retail_update['StockCode'] == '71270')]


In [ ]:
ecommerce_retail_update[(ecommerce_retail_update['InvoiceNo'] == '581538') & (ecommerce_retail_update['StockCode'] == '22992')]

In [ ]:
# Retain the last instance of a row if duplicates exist with varying quantities
ecommerce_retail_update = ecommerce_retail_update.drop_duplicates(subset=['InvoiceNo', 'StockCode', 'InvoiceDate', 'CustomerID'],keep='last')

In [ ]:
# Delete duplicate rows and retain only the first instance
ecommerce_retail_update = ecommerce_retail_update.drop_duplicates(keep='first')

In [ ]:
ecommerce_retail_update

**Nhận xét**

- Có 10038 hàng bị duplicated
- 2 hàng chỉ khác nhau về Quantity nguyên nhân đến từ việc ngay sau khi đặt hàng KH đã bấm thay đổi số lượng nhưng do hệ thống bị lỗi hay xảy ra vấn đề nên đã lưu thành 2 đơn hàng
- 2 hàng giống nhau hoàn toàn nguyên nhân do hệ thống bị lỗi nên dữ liệu đã bị duplicated

## **PART 2: Data Processing**

In [ ]:
# Create Recency, Frequency, and Monetary variables
# Take the last date as the baseline
Lastday = ecommerce_retail_update['day'].max()
# Create Cost column
ecommerce_retail_update['Spend'] = ecommerce_retail_update['Quantity'] * ecommerce_retail_update['UnitPrice']

# Calculate the Recency, Frequency, and Monetary values
rfm = ecommerce_retail_update.groupby('CustomerID').agg(
                                                        Recency = ('day', lambda x: -(Lastday - x.max()).days),
                                                        Frequency =('CustomerID', lambda x: x.count()) ,
                                                        Monetary = ('Spend', lambda x: x.sum())
                                                        ).reset_index()

# Adjust data types
rfm['Frequency'] = rfm['Frequency'].astype('int')
rfm.dtypes

In [ ]:
rfm.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure and a set of subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot the distribution of Recency
sns.boxplot(rfm['Recency'], ax=axes[0])
axes[0].set_title('Distribution of Recency')

# Plot the distribution of Frequency
sns.boxplot(rfm['Frequency'], ax=axes[1])
axes[1].set_title('Distribution of Frequency')

# Plot the distribution of Monetary
sns.boxplot(rfm['Monetary'], ax=axes[2])
axes[2].set_title('Distribution of Monetary')

# Adjust the layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Remove outliers
seventy_fifth = rfm['Recency'].quantile(0.75)
twenty_fifth = rfm['Recency'].quantile(0.25)
Recency_iqr = seventy_fifth - twenty_fifth
Recency_upper = seventy_fifth + (1.5 * Recency_iqr)
Recency_lower = twenty_fifth - (1.5 * Recency_iqr)


seventy_fifth = rfm['Frequency'].quantile(0.75)
twenty_fifth = rfm['Frequency'].quantile(0.25)
Frequency_iqr = seventy_fifth - twenty_fifth
Frequency_upper = seventy_fifth + (1.5 * Frequency_iqr)
Frequency_lower = twenty_fifth - (1.5 * Frequency_iqr)


seventy_fifth = rfm['Monetary'].quantile(0.75)
twenty_fifth = rfm['Monetary'].quantile(0.25)
Monetary_iqr = seventy_fifth - twenty_fifth
Monetary_upper = seventy_fifth + (1.5 * Monetary_iqr)
Monetary_lower = twenty_fifth - (1.5 * Monetary_iqr)

rfm_drop_outliers =rfm[
                       (rfm['Recency'] > Recency_lower) & (rfm['Recency'] < Recency_upper) &
                       (rfm['Frequency'] > Frequency_lower) & (rfm['Frequency'] < Frequency_upper) &
                       (rfm['Monetary'] > Monetary_lower)& (rfm['Monetary'] < Monetary_upper)
                       ]
rfm_drop_outliers.shape

In [ ]:
# Create a figure and a set of subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot the distribution of Recency
sns.boxplot(rfm_drop_outliers['Recency'], ax=axes[0])
axes[0].set_title('Distribution of Recency')

# Plot the distribution of Frequency
sns.boxplot(rfm_drop_outliers['Frequency'], ax=axes[1])
axes[1].set_title('Distribution of Frequency')

# Plot the distribution of Monetary
sns.boxplot(rfm_drop_outliers['Monetary'], ax=axes[2])
axes[2].set_title('Distribution of Monetary')

# Adjust the layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Use qcut to create R, M, F
rfm_drop_outliers['R'] = pd.qcut(rfm_drop_outliers['Recency'], 5, labels=range(1,6)).astype(str)
rfm_drop_outliers['F'] = pd.qcut(rfm_drop_outliers['Frequency'], 5, labels=range(1,6)).astype(str)
rfm_drop_outliers['M'] = pd.qcut(rfm_drop_outliers['Monetary'], 5, labels=range(1,6)).astype(str)
rfm_drop_outliers['RFM'] = rfm_drop_outliers['R'] + rfm_drop_outliers['F'] + rfm_drop_outliers['M']
rfm_drop_outliers.head()

In [ ]:
# Load data segmentation
segmentation = pd.read_csv(path + '/segmentation.csv')
segmentation.head()

In [ ]:
segmentation['RFM Score']= segmentation['RFM Score'].astype('string').str.split(',')
segmentation = segmentation.explode('RFM Score')
segmentation['RFM Score'] = segmentation['RFM Score'].apply(lambda x: x.replace(' ',''))
segmentation.head()

In [ ]:
# Merge segmentation
final_rfm = rfm_drop_outliers.merge(segmentation, left_on='RFM', right_on='RFM Score', how='left')
final_rfm.head()

**Mô tả quá trình**

- Tính Recency, Frequency và Monetary của từng khách hàng
- Loại bỏ Outliers
- Chia Recency, Frequency và Monetary thành 5 phân vị từ đó tính được điểm RFM của từng khách hàng
- Load file segmentation
- Kết hợp bảng rfm_drop_outliers và bảng segmentation bằng điểm RFM để phân loại khách hàng theo từng CustormerID

## **PART 3: Visualization**

### 3.1 Overall the distribution of the RFM Modelling

In [ ]:
import matplotlib.pyplot as plt
!pip install squarify
import squarify

In [ ]:
# Create a function for wrapping text
import textwrap
def wrap_text(text, width=12):  # Adjust width
    return "\n".join(textwrap.wrap(text, width=width))

In [ ]:
Custom_colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78', '#2ca02c', '#98df8a', '#d62728', '#ff9896', '#9467bd', '#c5b0d5', '#8c564b']

In [ ]:
# Users by segment
segment_usercnt = final_rfm[['Segment','CustomerID']].groupby(['Segment']).count().reset_index().rename(columns={'CustomerID':'user_cnt'})
# Percentage of users by segment
segment_usercnt['volumn_percent'] = ((segment_usercnt['user_cnt']/segment_usercnt['user_cnt'].sum())*100).round(0)
segment_usercnt['Segment'] = segment_usercnt['Segment'] + ' ' + segment_usercnt['volumn_percent'].astype(int).astype(str) + '%'
segment_usercnt

In [ ]:
# Calculate the average values
rfm_score_means = final_rfm.groupby('RFM Score').agg({
                                                      'Recency':'mean',
                                                      'Frequency':'mean',
                                                      'Monetary':'mean'
                                                      }).reset_index()


rfm_score_means = rfm_score_means.reset_index()

rfm_score_means.head()


In [ ]:
# Create subplots for each metric
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot histograms for each metric
sns.histplot(rfm_score_means['Recency'], ax=axes[0], bins=10, kde=True)
axes[0].set_title('Distribution of Recency')
axes[0].set_xlabel('Recency')
axes[0].set_ylabel('Custormers')

sns.histplot(rfm_score_means['Frequency'], ax=axes[1], bins=10, kde=True)
axes[1].set_title('Distribution of Frequency')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Custormers')

sns.histplot(rfm_score_means['Monetary'], ax=axes[2], bins=10, kde=True)
axes[2].set_title('Distribution of Monetary')
axes[2].set_xlabel('Monetary')
axes[2].set_ylabel('Custormers')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Use the text wrapping function to the Segment column
segment_usercnt['Segment_wrapped'] = segment_usercnt['Segment'].apply(wrap_text)

# Create treemap
squarify.plot(sizes=segment_usercnt['user_cnt'], label=segment_usercnt['Segment_wrapped'], color=Custom_colors, alpha=0.8,text_kwargs={'fontsize': 7})
plt.axis('off')
plt.show()

### 3.2 Distribution of RFM Modelling by time

In [ ]:
Custom_colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78', '#2ca02c', '#98df8a', '#d62728', '#ff9896', '#9467bd', '#c5b0d5', '#8c564b']

In [ ]:
# Segment user by time
rmf_customer_month = pd.merge(final_rfm, ecommerce_retail_update[['CustomerID','month']], on='CustomerID', how='left')
rmf_customer_month = rmf_customer_month.groupby(['month','Segment']).agg({'CustomerID':'count'}).reset_index().rename(columns={'CustomerID':'user_cnt'})
rmf_customer_month = rmf_customer_month.pivot_table(values='user_cnt', index='month', columns='Segment', fill_value =0)

# Create area chart
customer_month = rmf_customer_month.plot(kind='area', stacked=True, color=Custom_colors) # Rotate x-axis labels for better readability
plt.tight_layout() # Adjust layout to prevent overlapping
customer_month.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=2)
plt.xlabel("Month")
plt.ylabel("Number of Customers")
plt.title("Customer Segmentation by Month")
plt.show()

In [ ]:
# Segment recency by time
rmf_recency_month = pd.merge(final_rfm, ecommerce_retail_update[['CustomerID','month']], on='CustomerID', how='left')
rmf_recency_month = rmf_recency_month.groupby(['month','Segment']).agg({'Recency':'mean'}).reset_index()
rmf_recency_month = rmf_recency_month.pivot_table(values='Recency', index='month', columns='Segment', fill_value =0)

# Create area chart
recency_month = rmf_recency_month.plot(kind='area', stacked=True, color=Custom_colors)
plt.tight_layout() # Adjust layout to prevent overlapping
recency_month.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=2)
plt.xlabel("Month")
plt.ylabel("Average Recency (Days)")
plt.title("Average Recency by Customer Segment Over Time")
plt.show()

In [ ]:
# Segment frequency by time
rmf_frequency_month = pd.merge(final_rfm, ecommerce_retail_update[['CustomerID','month']], on='CustomerID', how='left')
rmf_frequency_month = rmf_frequency_month.groupby(['month','Segment']).agg({'Frequency':'mean'}).reset_index()
rmf_frequency_month = rmf_frequency_month.pivot_table(values='Frequency', index='month', columns='Segment', fill_value =0)

# Create area chart
frequency_month = rmf_frequency_month.plot(kind='area', stacked=True, color=Custom_colors)
plt.tight_layout() # Adjust layout to prevent overlapping
frequency_month.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=2)
plt.xlabel("Month")
plt.ylabel("Average Frequency")
plt.title("Average Frequency by Customer Segment Over Time")
plt.show()

In [ ]:
# Segment monetary by time
rmf_monetary_month = pd.merge(final_rfm, ecommerce_retail_update[['CustomerID','month']], on='CustomerID', how='left')
rmf_monetary_month = rmf_monetary_month.groupby(['month','Segment']).agg({'Monetary':'sum'}).reset_index()
rmf_monetary_month = rmf_monetary_month.pivot_table(values='Monetary', index='month', columns='Segment', fill_value =0)

# Create area chart
monetary_month = rmf_monetary_month.plot(kind='area', stacked=True, color=Custom_colors)
plt.ticklabel_format(axis='y', style='plain')
plt.tight_layout() # Adjust layout to prevent overlapping
monetary_month.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=2)
plt.xlabel("Month")
plt.ylabel("Total Monetary")
plt.title("Total Monetary by Customer Segment Over Time")
plt.show()

## **PART 4: Insight**

- Nhóm khách hàng chiếm tỉ trọng lớn nhất là: Hibernating Customers(17%), Champions(15%), Potential Loyalist(13%), At risk(12%)
- Chỉ số Recency cho thấy khách hàng quay trở lại mua hàng nhiều khoảng tháng 10 và 11, đặc biệt là số lượng khách hàng mới tăng đáng kể.
- Chỉ số Frequency cho thấy hầu hết các nhóm khách hàng có tần suất mua hàng giảm dần theo thời gian, đây là điều đang lo ngại.
- Chỉ số Monetary cho thấy doanh thu chủ yếu đến từ nhóm Champions, Loyal, Potential Loyal và tăng mạnh trong tháng 9,10,11.

## **PART 5: Recommendation**

- Tỉ lệ khách hàng Hibernating và At risk chiếm tương đối cao, công ty nên có các cuộc gọi và làm các cuộc khảo sát để tìm hiểu nguyên nhân tại sao họ ít mua hàng và từ đó khắc phục cũng như đưa ra các chương trình khuyến mãi hấp dẫn nhằm thu hút 2 nhóm khách hàng này mua hàng trở lại.
- Khách hàng có xu hướng mua hàng nhiều vào tháng 9,10,11 đây là thời gian họ trang trí nhà cửa, mua sắm đồ dùng cần thiết  để chuẩn bị cho dịp Giáng Sinh và Năm Mới. Vì vậy đây là khoảng thời gian tốt để công ty thu hút chuyển đổi nhóm khách hàng bằng các sản phẩm chất lượng, các vouchers hấp dẫn và đặc biệt cung cấp chất lượng dịch vụ tốt.
- Phân tích các nguyên nhân dẫn đến giảm tần suất mua hàng của các nhóm khách hàng, cải thiện các chiến dịch marketing và những chương trình đặc quyền dành cho nhóm khách hàng thân thiết.
- Công ty nên quan tâm vào chỉ số Recency nhất vì có thể đánh giá được tình hình kinh doanh của công ty và hiệu quả của các chiến dịch tiếp thị. Từ đó lên các chiến lược phù hợp tăng số lượng khách hàng và giữ chân khách hàng.